In [ ]:
import os
import geopandas as gpd
import pandas as pd
import warnings

In [ ]:
USER = '157412'
base_dir = f'C:\\Users\\{USER}\\OneDrive - UTS (1)\\ISF Projects - 23204_Quadrature_Mineral Availability for Renewables PEMMS\\3. Resources\\Graphite Production\\Sciencebase data'

COMMODITY_FILTER = 'graphite'

In [ ]:

def inspect_gdb(gdb_path):
    """Prints all layers and the columns of the first layer found."""
    layers = gpd.list_layers(gdb_path)
    print("--- Layers in GDB ---")
    print(layers[['name', 'geometry_type']])
    return layers['name'].tolist()

def load_gdb_region(gdb_path, alias_dict, region_name, commodity=COMMODITY_FILTER):
    """Load GDB layers, apply aliases, and filter for a specific commodity."""
    all_hits = []
    
    for layer_name, mapping in alias_dict.items():
        try:
            print(f"Processing {layer_name}...")
            gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
            gdf = gdf.rename(columns=mapping)
            
            # Only search columns likely to contain commodity info
            commodity_cols = [c for c in gdf.columns if any(kw in c.lower() for kw in 
                ['commodity', 'comm', 'deposit name', 'site name', 'facility name', 
                 'product', 'feature type', 'deposit type'])]
            search_cols = commodity_cols if commodity_cols else gdf.select_dtypes(include=['object']).columns
            
            mask = gdf[search_cols].astype(str).apply(
                lambda x: x.str.contains(commodity, case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Layer'] = layer_name
                hits['Region'] = region_name
                all_hits.append(hits)
                print(f"  --> Found {len(hits)} {commodity} records.")
            else:
                print(f"  --> No {commodity} found.")
                
        except Exception as e:
            print(f"  --> Error loading {layer_name}: {e}")

    if all_hits:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_hits, ignore_index=True)
    return None

def load_lac_data_and_aliases(folder_path, alias_dict, region_name='Latin America & Caribbean', commodity=COMMODITY_FILTER):
    """Load LAC CSV/Excel files, apply aliases, and filter for a specific commodity."""
    all_hits = []
    
    if not os.path.exists(folder_path):
        print(f"Error: Could not find folder at {folder_path}")
        return None

    files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls', '.csv'))]
    
    for file_name in files:
        layer_key = next((key for key in alias_dict.keys() if key in file_name), None)
        
        if not layer_key:
            continue
            
        try:
            print(f"Processing {file_name}...")
            
            file_path = os.path.join(folder_path, file_name)
            if file_name.endswith('.csv'):
                df = pd.read_csv(file_path, encoding='latin1')
            else:
                df = pd.read_excel(file_path)
            
            df = df.rename(columns=alias_dict[layer_key])
            
            # Only search columns likely to contain commodity info
            text_cols = df.select_dtypes(include=['object']).columns
            commodity_cols = [c for c in text_cols if any(kw in c.lower() for kw in 
                ['commodity', 'comm', 'deposit name', 'site name', 'facility name', 
                 'product', 'feature type', 'deposit type'])]
            search_cols = commodity_cols if commodity_cols else text_cols
            
            mask = df[search_cols].astype(str).apply(
                lambda x: x.str.contains(commodity, case=False, na=False)
            ).any(axis=1)
            
            hits = df[mask].copy()
            
            if not hits.empty:
                if 'DD Latitude' in hits.columns and 'DD Longitude' in hits.columns:
                    hits = hits.dropna(subset=['DD Latitude', 'DD Longitude'])
                    hits = gpd.GeoDataFrame(
                        hits, 
                        geometry=gpd.points_from_xy(hits['DD Longitude'], hits['DD Latitude']),
                        crs="EPSG:4326" 
                    )
                
                hits['Source_Layer'] = layer_key
                hits['Region'] = region_name
                all_hits.append(hits)
                print(f"  --> Found {len(hits)} {commodity} records.")
            else:
                print(f"  --> No {commodity} found.")
                
        except Exception as e:
            print(f"  --> Error: {e}")

    if all_hits:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_hits, ignore_index=True)
    return None

# 2. Loading GDBs per region

## 2.1. Africa

In [ ]:
# Path to Africa GDB
africa_path = os.path.join(base_dir, "Compilation of GIS Africa", "Africa_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(africa_path)

In [ ]:
AFRICA_ALIASES = {
    # --- AFR_Mineral_Facilities ---
    # 2,408 facilities in 1,866 locations from USGS 2018 Minerals Yearbook Vol. III
    # Data reflects state of minerals industries as of 2018 (Zambia: 2017)
    # LocConfid: 'A' (approximate), 'E' (exact)
    # FeatureTyp: 'Oil and Gas Fields', 'Mines and Quarries', 'Mineral Processing Plants',
    #   'Oil and Gas Refineries and (or) Petrochemical Complexes', 'Brine'
    # DsgAttr01 (Commodity Type): 'Fuel', 'Industrial', 'Metal', 'Fertilizer', 'Gemstone'
    # DsgAttr04 (Multiple Commodities): 'Y'/'N'
    # DsgAttr05 (Multiple Products): 'Y'/'N'
    # DsgAttr06 (Year): 2017–2018
    # DsgAttr07 (Annual Production Capacity): -999 = unknown; range 1–22,600,000
    # DsgAttr08 (Capacity Units): 'NA', 'Kilograms', 'Metric tons', 'Thousand metric tons',
    #   'Million 42-gallon barrels', 'Million cubic meters', 'Cubic meters',
    #   '42-gallon barrels per day', 'Thousand 42-gallon barrels',
    #   'Thousand 42-gallon barrels per day', 'Barrels of liquids per day',
    #   'Thousand bricks', 'Thousand carats'
    # DsgAttr09 (Capacity Notes): null=none, 'c'=combined, 'e'=estimated, 'g'=gross weight
    #   NOTE: lowercase codes (unlike China which uses uppercase C/E/G)
    # DsgAttr10 (Shared Capacity): lists FeatureUIDs sharing the same capacity
    # LocOpStat: 'Active', 'Assumed Active', 'C&M' (care & maintenance),
    #   'Closed', 'Construction', 'Development', 'Suspended'
    # Lat range: -35.31–37.26, Lon range: -23.73–57.61
    'AFR_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Accuracy',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Description',
        'FeatureTyp': 'Facility Type',
        'FeatureNam': 'Facility Name',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity Type',
        'DsgAttr02': 'Commodity',
        'DsgAttr03': 'Commodity Product',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Year',
        'DsgAttr07': 'Annual Production Capacity',
        'DsgAttr08': 'Capacity Units',
        'DsgAttr09': 'Capacity Notes',
        'DsgAttr10': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'MemoOther': 'Facility Comments',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'InfSource1': 'Mineral Facility Data Source',
        'LocSource1': 'GIS Data Source'
    },

    # --- AFR_Mineral_Deposits ---
    # Selected deposits from USGS PP 1802 and OFR 2005-1294
    # 8 commodity fields (DsgAttr01–08), plus Deposit Type and MRData ID
    'AFR_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'DsgAttr09': 'Deposit Type',
        'DsgAttr10': 'USGS MRData ID',
        'InfSource1': 'Data Source'
    },

    # --- AFR_Mineral_Exploration ---
    # Exploration sites from USGS World Minerals Exploration Database (WMED)
    # LocOpStat: variable/non-standardized text (not enumerated like China)
    # DsgAttr09 (Owner Type): variable text; empty cell = not available
    # DsgAttr10 (Owner Share): percentage equity share
    # DsgAttr11 (Project Active): year last verified active, range 1995–2018
    # DsgAttr12 (USGS WMED ID): internal WMED identifier
    # Lat range: -33.10–37.05, Lon range: -16.79–49.16
    'AFR_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'MemoLoc': 'Location Notes',
        'LocOpStat': 'Exploration Status',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'OwnerName': 'Owner Name',
        'DsgAttr09': 'Owner Type',
        'DsgAttr10': 'Owner Share',
        'DsgAttr11': 'Project Active',
        'DsgAttr12': 'USGS WMED ID',
        'InfSource1': 'Data Source'
    }
}

In [ ]:
africa_master = load_gdb_region(africa_path, AFRICA_ALIASES, 'Africa')

if africa_master is not None:
    display(africa_master.head(10))

## 2.2. China

In [ ]:
china_path = os.path.join(base_dir, "Compilation of GIS China", "CHN_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(china_path)

In [ ]:
CHINA_ALIASES = {
    # --- CHN_Mineral_Facilities ---
    # 2,424 mineral production/processing facilities from USGS 2019 Minerals Yearbook Vol. III
    # Data reflects state of minerals industries as of 2019
    # DsgAttr03 (Multiple Commodities): 'Y'/'N' - multiple commodities at one facility
    # DsgAttr04 (Multiple Products): 'Y'/'N' - multiple products at one facility
    # DsgAttr06 (Annual Production Capacity): -999 = unknown/unavailable; range 1~130,000
    # DsgAttr07 (Capacity Units): 
        #   'NA', 'kg' (kilograms), 'mt' (metric tons), 'MMbbl' (Million 42-gal barrels),
        #   'mcm' (Million cubic meters), 'million mt', 'kmt' (thousand mt)
    # DsgAttr08 (Capacity Notes): 
        # null=none, 'C'=combined facilities, 'E'=estimated, 'G'=gross weight
    # DsgAttr09 (Shared Capacity): lists FeatureUIDs of other facilities sharing the same capacity
    'CHN_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Confidence',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'FeatureTyp': 'Feature Type',
        'FeatureNam': 'Facility Name',
        'Label': 'Label',
        'DsgAttr01': 'Commodity',
        'DsgAttr02': 'Commodity Product',
        'DsgAttr03': 'Multiple Commodities',
        'DsgAttr04': 'Multiple Products',
        'DsgAttr05': 'Product Notes',
        'DsgAttr06': 'Annual Production Capacity',
        'DsgAttr07': 'Capacity Units',
        'DsgAttr08': 'Capacity Notes',
        'DsgAttr09': 'Shared Capacity',
        'LocOpStat': 'Operating Status',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'InfSource1': 'Mineral Facility Data Source',
        'LocSource1': 'GIS Data Source'
        },

    # --- CHN_Mineral_Deposits ---
    # Selected deposits from USGS OFR 2005-1294 and Professional Paper 1802
    # Only 4 commodity fields (DsgAttr01–04), unlike Exploration which has 7
    # DsgAttr05 is Deposit Type (not Commodity 5)
    # Note: COUNTRY, LATITUDE, LONGITUDE are UPPERCASE (unlike other layers)
    # Lat range: 18.7166–53, Lon range: 81.2499–132.4833
    # Label1 is just FeatureUID repeated for map labeling
    'CHN_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'LATITUDE': 'DD Latitude',
        'LONGITUDE': 'DD Longitude',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Deposit Type',
        'Label1': 'Label',
        'InfSource1': 'Data Source',
    },

    # --- CHN_Mineral_Exploration ---
    # 386 exploration sites from USGS World Minerals Exploration Database (WMED)
    # LocOpStat valid values: 'Exploration', 'Feasibility Study', 'Inactive', 'Pre-production'
    # LocConf valid values: 'A' (approximated), 'E' (exact)
    # DsgAttr09 (Source Year) range: 1994–2022
    # Lat range: 21.48–53.0082, Lon range: 74.26–132.31
    # Note: admin field is AMD1 (not ADM1 like other layers)
    # Low-accuracy locations use ADM1 centroid coordinates
    'CHN_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Location Description',
        'Country': 'Country (Short Form)',
        # Note THAT THERE AS A TYPO IN THE ORIGINAL DATA. Hence the "AMD" instead of "ADM"
        'AMD1': 'AMD1 Name',
        'LocOpStat': 'Exploration Status',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Confidence',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OwnerName': 'Owner Name',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Operation Source 1',
        'InfSource4': 'Operation Source 2',
        'InfSource5': 'Company Source 1',
        'InfSource6': 'Company Source 2',
        'InfSource7': 'Commodity Source 1',
        'InfSource8': 'Commodity Source 2',
    }
}

In [ ]:
china_master = load_gdb_region(china_path, CHINA_ALIASES, 'China')

if china_master is not None:
    display(china_master.head(10))

## 2.3. Indopacific

In [ ]:
indopac_path = os.path.join(base_dir, "Compilation of GIS Indopacific", "INDOPAC_GIS.gdb.zip")

# Inspect
layers = inspect_gdb(indopac_path)

In [ ]:
INDOPAC_ALIASES = {
    # --- INDOPAC_Mineral_Facilities ---
    # Fuel and non-fuel mineral production/processing facilities in Indo-Pacific nations
    # FeatureTyp: 'Brine', 'Mineral Processing Plants', 'Mines and Quarries',
    #   'Oil and Gas Fields', 'Oil and Gas Refineries and (or) Petrochemical Complexes'
    # NOTE: No 'Commodity Type' field (unlike Africa) — DsgAttr01 is Commodity directly
    # DsgAttr04 (Multiple Commodities): 'Y'/'N'
    # DsgAttr05 (Multiple Products): 'Y'/'N'
    # DsgAttr06 (Annual Production Capacity): -999 = unknown; range 1–110,000
    # DsgAttr07 (Capacity Units): 'NA', 'kbbl', 'kg', 'kmt', 'mcm',
    #   'million mt', 'MMbbl', 'mt', 'Thousand carats'
    # DsgAttr08 (Capacity Notes): Null=none, 'C'=combined, 'E'=estimated, 'G'=gross weight
    # DsgAttr09 (Shared Capacity): lists FeatureUIDs sharing the same capacity
    # LocOpStat: 'Assumed Active', 'C&M' (care & maintenance), 'Suspended'
    # LocConf: 'A' (approximate), 'E' (exact)
    # Lat range: -46.59–50.32, Lon range: 79.82–178.72
    # NOTE: uses LocConf (not LocConfid), Label (not Label1)
    # Has 6 equity owners (OwnerName1–6), more than other regions
    'INDOPAC_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Facility Name',
        'FeatureTyp': 'Feature Type',
        'DsgAttr01': 'Commodity',
        'DsgAttr02': 'Commodity Product',
        'DsgAttr03': 'Product Notes',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Annual Production Capacity',
        'DsgAttr07': 'Capacity Units',
        'DsgAttr08': 'Capacity Notes',
        'DsgAttr09': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'DsgAttr10': 'Location Description',
        'ADM1': 'ADM1 Name',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'OwnerName6': 'Equity Owner 6',
        'Label': 'Label',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Capacity Source 1',
        'InfSource4': 'Capacity Source 2',
        'InfSource5': 'Company Source 1',
        'InfSource6': 'Company Source 2',
        'InfSource7': 'Other Source',
        'MemoSource': 'Source notes',
    },

    # --- INDOPAC_Mineral_Development ---
    # 285 development sites across Indo-Pacific nations (Bangladesh, Bhutan, Brunei,
    #   Burma, Fiji, Malaysia, Mongolia, New Caledonia, New Zealand, Papua New Guinea,
    #   Philippines, Singapore, Solomon Islands, South Korea, Sri Lanka, Taiwan, Vietnam)
    # NOTE: Only 6 commodity fields (DsgAttr01–06), then DsgAttr07 = Commodity Product
    # DsgAttr08 = Location Description (NOT Commodity 8)
    # FeatureTyp: 'Mineral Processing Plants', 'Mines and Quarries',
    #   'Oil and Gas Fields', 'Oil and Gas Refineries and (or) Petrochemical Complexes'
    # LocOpStat: 'Construction', 'Feasibility', 'Inactive', 'Planned Construction', 'Pre-Production'
    # LocConf: 'A' (approximate), 'E' (exact)
    # DsgAttr09 (Source Year): range 2014–2023
    # Lat range: -43.50–49.56, Lon range: 80.50–179.27
    # Has OperateNam + OwnerName1–5 (not single OwnerName like Africa Exploration)
    'INDOPAC_Mineral_Development': {
        'FeatureUID': 'Development ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'FeatureTyp': 'Feature Type',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity Product',
        'LocOpStat': 'Development Status',
        'DsgAttr08': 'Location Description',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'Label': 'Label',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Commodity Source 1',
        'InfSource4': 'Commodity Source 2',
        'InfSource5': 'Operating Stage Source 1',
        'InfSource6': 'Operating Stage Source 2',
        'InfSource7': 'Company Source 1',
        'InfSource8': 'Company Source 2',
        'InfSource9': 'Other Source',
        'MemoSource': 'Source notes',
    },

    # --- INDOPAC_Mineral_Exploration ---
    # Exploration sites from USGS WMED across Indo-Pacific nations
    # NOTE: Only 6 commodity fields (DsgAttr01–06), then DsgAttr07 = Location Description
    # LocOpStat: 'Exploration', 'Inactive', 'Preliminary Study'
    # LocConf: 'A' (approximate), 'E' (exact)
    # DsgAttr09 (Source Year): range 1998–2023
    # Lat range: -46.55–50.55, Lon range: 78.96–179.96
    # Has OperateNam + OwnerName1–4 (4 owners, not 5 like Development)
    # No DsgAttr08 — Location Description is DsgAttr07 here (vs DsgAttr08 in Development)
    'INDOPAC_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'LocOpStat': 'Exploration Status',
        'DsgAttr07': 'Location Description',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Commodity Source 1',
        'InfSource4': 'Commodity Source 2',
        'InfSource5': 'Operating Stage Source 1',
        'InfSource6': 'Operating Stage Source 2',
        'InfSource7': 'Company Source 1',
        'InfSource8': 'Company Source 2',
        'InfSource9': 'Other Source',
        'Label': 'Label',
    },
}

In [ ]:
indopac_master = load_gdb_region(indopac_path, INDOPAC_ALIASES, 'Indopacific')

if indopac_master is not None:
    display(indopac_master.head(10))

## 2.4. SW Asia

In [ ]:
# Path to SW Asia GDB
swa_path = os.path.join(base_dir, "Compilation of GIS SW Asia", "SWAsia_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(swa_path)

In [ ]:
# Note the intentional typo in 'SWA_Mineral_Facilties' to match the database layer name
SWA_ALIASES = {
    # --- SWA_Mineral_Facilties --- (typo is in original data)
    # Fuel and non-fuel mineral production/processing facilities in SW Asia
    # FeatureTyp: 'Oil and Gas Fields', 'Mines and Quarries', 'Mineral Processing Plants',
    #   'Oil and Gas Refineries and (or) Petrochemical Complexes', 'Brine'
    # DsgAttr01 (Commodity Type): 'Fuel', 'Industrial', 'Metal', 'Fertilizer', 'Gemstone'
    # DsgAttr04 (Multiple Commodities): 'Y'/'N'
    # DsgAttr05 (Multiple Products): 'Y'/'N'
    # DsgAttr06 (Year): 2018 only
    # DsgAttr07 (Annual Production Capacity): -999.999 = unknown; range 0.75–12,000
    # DsgAttr08 (Capacity Units): 'NA' or variable text
    # DsgAttr09 (Capacity Notes): null=none, 'c'=combined, 'e'=estimated, 'g'=gross weight
    #   NOTE: lowercase codes (like Africa, unlike China)
    # LocOpStat: 'Assumed Active', 'C&M' (care & maintenance), 'Development'
    # LocConfid: 'A' (approximate), 'E' (exact)
    # Only 4 equity owners (no OwnerName5)
    # Lat range: -10.24–42.94, Lon range: 45.03–138.00
    'SWA_Mineral_Facilties': {
        'FeatureUID': 'USGS Facility ID',
        'Label1': 'Label',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Facility Name',
        'FeatureTyp': 'Facility Type',
        'DsgAttr01': 'Commodity Type',
        'DsgAttr02': 'Commodity',
        'DsgAttr03': 'Commodity Product',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Year',
        'DsgAttr07': 'Annual Production Capacity',
        'DsgAttr08': 'Capacity Units',
        'DsgAttr09': 'Capacity Notes',
        'DsgAttr10': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'MemoLoc': 'Location Description',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Accuracy',
        'LocSource1': 'GIS Data Source',
        'InfSource1': 'Mineral Facility Data Source',
    },

    # --- SWA_Mineral_Deposits ---
    # Selected mineral deposits in SW Asia
    # NOTE: 11 commodity fields (DsgAttr01–11), zero-padded aliases (Commodity 01, 02, etc.)
    # DsgAttr12 = Deposit Type, DsgAttr13 = Critical Commodity ('Yes')
    # Only 1 InfSource field
    # Lat range: -8.96–40.67, Lon range: 44.65–140.45
    'SWA_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        # I've changed these to non-zero padded to match the other regions:
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'DsgAttr09': 'Commodity 9',
        'DsgAttr10': 'Commodity 10',
        'DsgAttr11': 'Commodity 11',
        'DsgAttr12': 'Deposit Type',
        'DsgAttr13': 'Critical Commodity',
        'InfSource1': 'Data Source',
    },

    # --- SWA_Mineral_Exploration ---
    # Exploration sites in SW Asia
    # NOTE: 9 commodity fields (DsgAttr01–09), more than most other regions
    # LocOpStat: 'Exploration', 'Feasibility Study', 'Inactive',
    #   'Pre-Production', 'Reserves Development'
    # No OwnerName, no Owner Type/Share/Project Active/WMED ID fields
    # Only 1 InfSource field
    # Lat range: -10.23–41.00, Lon range: 46.51–140.86
    'SWA_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'LocOpStat': 'Exploration Status',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'DsgAttr09': 'Commodity 9',
        'MemoLoc': 'Location Notes',
        'Label1': 'Label',
        'InfSource1': 'Data Source',
    },
}

In [ ]:
swa_master = load_gdb_region(swa_path, SWA_ALIASES, 'SW Asia')

if swa_master is not None:
    display(swa_master.head(10))

## 2.5. Latin American and Caribbean

In [ ]:
# Path to LAC folder
lac_path = os.path.join(base_dir, "Compilation of GIS Latin American and Carib")

In [ ]:
# The column headers in the LAC CSV/Excel files are different from the GDBs.
# We map them to match the aliases used in Africa, China, and Indopacific.
# NOTE: LAC is an older dataset (2016 publication, 2013–2015 data) with a simpler
# flat CSV schema — no DsgAttr numbering, different column naming conventions.
LAC_ALIASES = {
    # --- MINFAC_LAC ---
    # 1,650 mineral facilities across Latin America and the Caribbean (as of 2014)
    # Data from USGS Minerals Yearbook Vol. III
    # COMMTYPE: 'Fuel', 'Industrial', 'Metal' (no 'Fertilizer' or 'Gemstone' unlike Africa/SWA)
    # FACTYPE: 'B' (brine), 'M' (mine), 'MP' (mine & plant), 'P' (plant),
    #   'OG' (oil & gas fields), 'OR' (oil & gas refineries)
    # STATUS: 'A' (active), 'C' (closed), 'CM' (care & maintenance),
    #   'S' (suspended), 'U' (unknown), 'UC' (under construction), 'UD' (under development)
    # MULTI_COMM / MULTI_PROD: 'Y' flag only (blank = no)
    # ECAP: 'e' = estimated capacity
    # COMBOCAP: shared capacity flag, format "***#-#" listing FACID_NUMs sharing capacity
    # UNITS: 'NA', 'kg', 'mt', 'tmt' (thousand mt), 'm42-gb' (M 42-gal barrels),
    #   't42-gb' (thousand 42-gal barrels), 't42-gbpd' (thousand 42-gal barrels/day),
    #   'mcm' (M cubic meters), 'tcm' (thousand cubic meters), 'tcmpd' (thousand cm/day)
    # LOCACC: 'A' (approximate), 'E' (exact)
    # YEAR: 2013–2015
    # Lat range: -53.36–31.85, Lon range: -116.58–-34.85
    'MINFAC_LAC': {
        'FACID': 'USGS Facility ID',
        'DDLAT': 'DD Latitude',
        'DDLONG': 'DD Longitude',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'LOCNAME': 'Facility Name',
        'LOCDESC': 'Location Description',
        'COMMTYPE': 'Commodity Type',
        'COMMODITY': 'Commodity',
        'MIN_PROD': 'Commodity Product',
        'FACTYPE': 'Facility Type',
        'MULTI_COMM': 'Multiple Commodities',
        'MULTI_PROD': 'Multiple Products',
        'YEAR': 'Year',
        'OPERATOR': 'Major Operating Company',
        'OWNER': 'Equity Owner 1',
        'ANNCAP': 'Annual Production Capacity',
        'ECAP': 'Estimated Capacity',
        'UNITS': 'Capacity Units',
        'COMBOCAP': 'Shared Capacity',
        'STATUS': 'Facility Status',
        'LOCACC': 'Location Accuracy',
        'COMMENTS': 'Facility Comments',
        'SOURCEID': 'Source ID',
    },

    # --- EXPLORE_LAC ---
    # 161 exploration/development sites across Latin America and the Caribbean
    # COMM: commodities often comma-separated in one column (unlike other regions)
    # NUM_COMM: count of commodities at the site, range 1–5
    # PROJTYPE: 'D' (development), 'E' (active exploration),
    #   'F' (feasibility), 'P' (exploration at producing mine / mine expansion)
    # OPERATOR is the operating company (not the owner)
    # OWNER is the parent/equity holder (includes ownership % if known)
    # LOCACC: 'A' (approximate), 'E' (exact)
    # YEAR: 2013–2015
    # Lat range: -53.00–31.36, Lon range: -111.91–-34.95
    'EXPLORE_LAC': {
        'FACID': 'Exploration ID',
        'DDLAT': 'DD Latitude',
        'DDLONG': 'DD Longitude',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'PROJNAME': 'Site Name',
        'COMM': 'Commodity',
        'NUM_COMM': 'Number of Commodities',
        'YEAR': 'Year',
        'OPERATOR': 'Major Operating Company',
        'OWNER': 'Equity Owner 1',
        'LOCACC': 'Location Accuracy',
        'PROJTYPE': 'Exploration Status',
        'SOURCEID': 'Source ID',
    },
}

In [ ]:
lac_master = load_lac_data_and_aliases(lac_path, LAC_ALIASES)

if lac_master is not None:
    display(lac_master.head(10))

# 3. Setting up output dataframes

In [ ]:
# --- FINAL MERGE AND EXPORT CELL ---

# 1. Combine all regional GeoDataFrames
master_datasets = [africa_master, china_master, indopac_master, swa_master, lac_master]

# Filter out any regions that might have returned None (if no graphite was found)
valid_datasets = [df for df in master_datasets if df is not None]

# Concatenate into one massive global spatial dataset
global_graphite_gdf = pd.concat(valid_datasets, ignore_index=True)

print(f"\n--- GLOBAL MERGE COMPLETE ---")
print(f"Total Graphite Records Identified: {len(global_graphite_gdf)}")

# 2. Reorder columns to put the most important ones first for easier reading
front_cols = [
    'Region', 'Source_Layer', 'Country (Short Form)', 'Facility Name', 'Site Name', 'Deposit Name',
    'Commodity Type', 'Commodity', 'Commodity 1', 'Facility Status', 'Exploration Status',
    'Annual Production Capacity', 'Capacity Units', 'Major Operating Company', 'Owner Name'
]

# Get remaining columns that exist in the dataframe
remaining_cols = [c for c in global_graphite_gdf.columns if c not in front_cols and c != 'geometry']

# Ensure we only try to sort columns that actually exist in the merged dataset
front_cols_existing = [c for c in front_cols if c in global_graphite_gdf.columns]

# Reapply column order, always keeping geometry at the end for spatial formats
final_cols = front_cols_existing + remaining_cols + ['geometry']
global_graphite_gdf = global_graphite_gdf[final_cols]

# 3. Export to CSV for Excel/Tabular Analysis
csv_export_path = os.path.join(base_dir, "Global projects - Clean data", "Global_Graphite_Projects.csv")
# Drop the geometry column for the pure CSV export so it opens cleanly in Excel
global_graphite_gdf.drop(columns='geometry').to_csv(csv_export_path, index=False)
print(f"Saved Tabular Data to: {csv_export_path}")

# 4. Export to a Spatial Format (Shapefile or GeoJSON) for QGIS/ArcGIS
geojson_export_path = os.path.join(base_dir, "Global projects - Clean data", "Global_Graphite_Projects.geojson")
# GeoJSON is generally safer and less restrictive with column names/lengths than Shapefiles
global_graphite_gdf.to_file(geojson_export_path, driver='GeoJSON')
print(f"Saved Spatial Data to: {geojson_export_path}")

# Quick preview
display(global_graphite_gdf.head())

In [ ]:
global_graphite_gdf.to_csv('outputs/Global_Graphite_Projects.csv', index=False)